In [ ]:
EXPECTED={'complete': True, 'chunks': 270405, 'parts': 3, 'new_model_forward_inputs': 0, 'vectors_bitwise_preserved': True, 'all_ids_match_corpus_part_order': True, 'files': {'embed_manifest.json': {'bytes': 2414, 'sha256': 'd06f12900e73befe942c2289e44c36f9c62572728fecffcdc60de69f64cec207'}, 'vectors/zh-part-001.npy': {'bytes': 893199488, 'sha256': 'ed310b8cbe867711174612f078993a244971dc15b51a9d04d8416e91333dbe63'}, 'vectors/en-part-001-ids.parquet': {'bytes': 692, 'sha256': '8c7ac9224183e872872151a3f8f2de31c433109c24b74739ecdbd998ee3bac25'}, 'vectors/zh-part-001-ids.parquet': {'bytes': 1294525, 'sha256': '04fe3a8058bf82576af8f237535664f4b6d5c8ef308ab2062bcfd28010b6c084'}, 'vectors/en-part-001.npy': {'bytes': 71808, 'sha256': '2920534d06a89b547f5a4337b859cb9969a2a59ca8416e2e17759c7b1f3736b6'}, 'vectors/vi-part-007.npy': {'bytes': 491202688, 'sha256': '6bce006d943e0e600bf5ecbb604c158ae476a6addbc8052a1ef3a3f3e7f37b76'}, 'vectors/vi-part-007-ids.parquet': {'bytes': 526152, 'sha256': 'fc9d06267e6eb98c65a73f35f174bcd8ba75172c4480ced025e089d6b368eed0'}}, 'archive': {'bytes': 1386298683, 'sha256': '47d937fff2ea5ee3e80618a1a2a4e5ceb4f51f023bd25bc32ecd504edbcc62f1'}}
from pathlib import Path
import hashlib,json
import numpy as np
import pyarrow.parquet as pq
roots=list(Path('/kaggle/input').rglob('embed_manifest.json'))
corpora=list(Path('/kaggle/input').rglob('chunk_manifest.json'))
assert len(roots)==len(corpora)==1
root=roots[0].parent;corpus=corpora[0].parent
actual={p.relative_to(root).as_posix():p for p in root.rglob('*') if p.is_file()}
assert set(actual)==set(EXPECTED['files'])
for name,meta in EXPECTED['files'].items():
    with actual[name].open('rb') as f:digest=hashlib.file_digest(f,'sha256').hexdigest()
    assert actual[name].stat().st_size==meta['bytes'] and digest==meta['sha256'],name
manifest=json.loads((root/'embed_manifest.json').read_text())
with (corpus/'chunk_manifest.json').open('rb') as f:corpus_sha=hashlib.file_digest(f,'sha256').hexdigest()
assert corpus_sha==manifest['corpus_manifest_sha256']
assert manifest['complete'] and manifest['chunks']==270405 and manifest['parts_done']==3
assert manifest['new_model_forward_inputs']==0 and manifest['vectors_bitwise_preserved']
counts={};all_ids=set()
for lang,part in [('vi','007'),('en','001'),('zh','001')]:
    key=f'{lang}-part-{part}'
    ids=pq.read_table(root/f'vectors/{key}-ids.parquet',columns=['chunk_id'])['chunk_id'].to_pylist()
    expected=pq.read_table(corpus/f'chunks/{lang}/part-{part}.parquet',columns=['chunk_id'])['chunk_id'].to_pylist()
    assert ids==expected and not all_ids.intersection(ids);all_ids.update(ids)
    v=np.load(root/f'vectors/{key}.npy',mmap_mode='r',allow_pickle=False)
    assert v.dtype==np.float16 and v.shape==(len(ids),2560)
    for start in range(0,len(v),8192):
        b=np.array(v[start:start+8192],dtype=np.float32)
        assert np.isfinite(b).all() and np.max(abs(np.linalg.norm(b,axis=1)-1))<0.01
    counts[lang]=len(ids);del v
assert counts=={'vi':95938,'en':14,'zh':174453} and len(all_ids)==270405
result=dict(complete=True,dataset='ntklinhfitus/medxir-vectors-qwen3-4b-additions',version=1,
    corpus='ntklinhfitus/medxir-chunk-corpus-v2',rows=len(all_ids),rows_by_language=counts,
    all_file_checksums_verified=True,chunk_ids_and_order_match=True,all_vectors_finite_and_unit_norm=True,
    vectors_bitwise_unchanged_from_verified_source=True,new_model_forward_inputs=0)
Path('/kaggle/working/published_vectors_validation.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))
